# Product Confirmation Workflow

This notebook downloads DIST-ALERT products from S3, unzips them, and runs the confirmation workflow.

In [2]:
import pandas as pd
import shutil
import zipfile
import requests
from pathlib import Path
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor
from dist_s1 import run_sequential_confirmation_of_dist_products_workflow
from utils import unzip_dist_s1_prod, wrap_run_sequential_confirmation_of_dist_products_workflow
import multiprocessing
from functools import partial


/Users/cmarshak/miniforge3/envs/dist-s1-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
token = 'dist-event'

In [4]:
tmp_dir =  Path(f'tmp_{token}')
unconfirmed_products_dir =  Path(f'unconfirmed_products_{token}')
confirmed_products_dir =  Path(f'confirmed_products_{token}')

tmp_dir.mkdir(exist_ok=True)
unconfirmed_products_dir.mkdir(exist_ok=True)
confirmed_products_dir.mkdir(exist_ok=True)

In [5]:
# Load the test products CSV
csv_path = Path('dist-s1-events_october-17-2025.csv')
df = pd.read_csv(csv_path)
df.head()

,job_name,zip_url,browse_url,product_request_time,processing_duration,high_confidence_alert_threshold,mgrs_tile_id,post_date_buffer_days,stride_for_norm_param_estimation,n_workers_for_norm_param_estimation,...,model_source,memory_strategy,batch_size_for_norm_param_estimation,post_date,track_number,low_confidence_alert_threshold,n_workers_for_despeckling,device,max_pre_imgs_per_burst_mw,model_compilation
0,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,1008.556,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-07-30,144,2.5,4,best,none,False
1,durkee_fire_2024__11TMK,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,500.253,4.5,11TMK,1,7,4,...,transformer_optimized,high,32,2024-09-09,42,2.5,4,best,none,False
2,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,1468.343,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-08-23,144,2.5,4,best,none,False
3,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,651.496,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-09-11,71,2.5,4,best,none,False
4,durkee_fire_2024__11TMJ,https://hyp3-tibet-jpl-test-contentbucket-hrat...,https://hyp3-tibet-jpl-test-contentbucket-hrat...,2025-10-16T22:25:24+00:00,667.371,4.5,11TMJ,1,7,4,...,transformer_optimized,high,32,2024-06-29,42,2.5,4,best,none,False


In [6]:
def download_file(url, destination_path):
    dst_dir = destination_path.parent
    dst_dir.mkdir(exist_ok=True, parents=True)

    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    with open(destination_path, 'wb') as file:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                file.write(chunk)
    
    return destination_path


In [7]:
download_tasks = []
for _, row in df.iterrows():
    url = row['zip_url']
    filename = Path(url).name
    
    # Customize!!!!
    job_name = row['job_name']
    dist_event = job_name.split('__')[0]
    dst_dir = tmp_dir / dist_event
    zip_path = dst_dir / filename

    download_tasks.append((url, zip_path))

In [8]:
download_file_p = lambda t: download_file(*t)
with ThreadPoolExecutor(max_workers=10) as executor:
    paths = list(tqdm(executor.map(download_file_p, download_tasks[:]), total=len(download_tasks)))

  0%|                                   | 0/876 [00:00<?, ?it/s]


HTTPError: 404 Client Error: Not Found for url: https://hyp3-tibet-jpl-test-contentbucket-hratibh1y9pa.s3.us-west-2.amazonaws.com/a4771a64-4f2c-4e1d-9e8a-e04ec2c7dccf/OPERA_L3_DIST-ALERT-S1_T11TMJ_20240730T135819Z_20251017T014844Z_S1A_30_v0.1.zip

# Unzip

In [9]:
downloaded_zips = list(tmp_dir.rglob('*.zip'))
len(downloaded_zips)

876

In [14]:
num_processes = 8 
print('Total processes: ', multiprocessing.cpu_count())
print(f"Using {num_processes} processes for unzipping.")


with multiprocessing.Pool(processes=num_processes) as pool:
    unzipper = partial(unzip_dist_s1_prod, unconfirmed_products_dir=unconfirmed_products_dir)

    results = pool.imap(unzipper, downloaded_zips[434 + 47:])
    for _ in tqdm(results, total=len(downloaded_zips), desc="Unzipping Files"):
        pass

Total processes:  12
Using 8 processes for unzipping.


Unzipping Files:  45%|██████████████████▍                      | 395/876 [00:24<00:29, 16.35it/s]


In [15]:
subdirs = list(unconfirmed_products_dir.rglob('OPERA_L3_DIST-ALERT-S1*/'))
mgrs_tiles_unzipped = list(set([subdir.parent.name for subdir in subdirs]))
mgrs_tiles_unzipped[:3]

['19HBD', '10TEK', '49MDN']

In [16]:
subdirs = list(unconfirmed_products_dir.rglob('OPERA_L3_DIST-ALERT-S1*/'))
mgrs_ts_unzipped_dirs = list(set([subdir.parent for subdir in subdirs]))
mgrs_ts_unzipped_dirs[:3]

[PosixPath('unconfirmed_products_dist-event/brazzaville_flood_and_landslides_2024/33MWR'),
 PosixPath('unconfirmed_products_dist-event/tlacotalpan_flood_2024/14QRF'),
 PosixPath('unconfirmed_products_dist-event/southwest_france_flood_2023/30TXR')]

In [17]:
# cleanup_temp = True
# if cleanup_temp:
#     shutil.rmtree(tmp_dir)

# Confirmation

In [18]:
# confirm_kwargs = {'alert_low_conf_thresh': 3, 
#                   'alert_high_conf_thresh': 5, 
#                   'percent_reset_thresh': 50, 
#                   'no_day_limit': 18, 
#                   'no_count_reset_thresh': 4}

In [19]:
# confirmer = partial(wrap_run_sequential_confirmation_of_dist_products_workflow, 
#                         unconfirmed_products_dir=unconfirmed_products_dir, 
#                         confirmed_products_dir=confirmed_products_dir,
#                         confirm_kwargs=confirm_kwargs
#                        )
# list(map(confirmer, mgrs_ts_unzipped_dirs_f[:1]))

In [20]:
with multiprocessing.Pool(processes=5) as pool:
    confirmer = partial(wrap_run_sequential_confirmation_of_dist_products_workflow, 
                        unconfirmed_products_dir=unconfirmed_products_dir, 
                        confirmed_products_dir=confirmed_products_dir,
                        #confirm_kwargs=confirm_kwargs
                       )

    results = pool.imap(confirmer, mgrs_ts_unzipped_dirs[4:])
    for _ in tqdm(results, total=len(mgrs_ts_unzipped_dirs), desc="Confirming Products"):
        pass

Confirming 39 products:  87%|█▋| 34/39 [04:00<00:39,  7.91s/it]/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:248: UserWarning: Layer GEN-DIST-STATUS does not exist at path: unconfirmed_products_dist-event/durkee_fire_2024/11TMK/OPERA_L3_DIST-ALERT-S1_T11TMK_20240904T135804Z_20251017T014400Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11TMK_20240904T135804Z_20251017T014400Z_S1A_30_v0.1_GEN-DIST-STATUS.tif
  warn(f'Layer {layer} does not exist at path: {path}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_models.py:248: UserWarning: Layer GEN-METRIC does not exist at path: unconfirmed_products_dist-event/durkee_fire_2024/11TMK/OPERA_L3_DIST-ALERT-S1_T11TMK_20240904T135804Z_20251017T014400Z_S1A_30_v0.1/OPERA_L3_DIST-ALERT-S1_T11TMK_20240904T135804Z_20251017T014400Z_S1A_30_v0.1_GEN-METRIC.tif
  warn(f'Layer {layer} does not exist at path: {path}', UserWarning)
/Users/cmarshak/bekaert-team/dist-s1/src/dist_s1/data_models/output_mod

ValueError: Product directory missing required layers: unconfirmed_products_dist-event/durkee_fire_2024/11TMK/OPERA_L3_DIST-ALERT-S1_T11TMK_20240904T135804Z_20251017T014400Z_S1A_30_v0.1